In [29]:
import pandas as pd 
from pathlib import Path
import matplotlib_inline as plt

Panel A. Stablecoin sector (transmitters) 
- On-chain flows (Hourly/daily) (2020-latest)
    - Net print/burn USDT , USDC , DAI
    - aggregate stable coin supply
- Peg devataions (Hourly/daily) (2020-latest)
    - Secondary market prive vs $1 peg USDT , USDC , DAI
    - redemption-pressure index(constucted) USDT , USDC , DAI

In [32]:
# Panel A. Stablecoin sector (transmitters)
from pathlib import Path
import pandas as pd

PANEL_A_DATA_FOLDER = Path('Panel_A_Data')

# On-chain flows (Daily, 2020-latest)
net_mint_burn_data = pd.read_csv(
    PANEL_A_DATA_FOLDER / 'Net_Mint_Burn_USDT_USDC_DAI_Daily.csv', parse_dates=['Date'])
OC_flow_USDC = net_mint_burn_data[['Date', 'USDC_Net_Mint_Burn_Units']].copy()
OC_flow_DAI = net_mint_burn_data[['Date', 'DAI_Net_Mint_Burn_Units']].copy()
OC_flow_USDT = net_mint_burn_data[['Date', 'USDT_Net_Mint_Burn_Units']].copy()

# Aggregate stablecoin supply
aggregate_supply_data = pd.read_csv(
    PANEL_A_DATA_FOLDER / 'Aggregate_Stablecoin_Supply_Daily.csv', parse_dates=['Date'])
Aggregate_stablecoin_supply = aggregate_supply_data[[
    'Date', 'Aggregate_Stablecoin_Supply']].copy()

# Secondary-market prices and peg deviations versus $1
peg_price_data = pd.read_csv(
    PANEL_A_DATA_FOLDER / 'Secondary_Prices_and_Peg_Deviations_Daily.csv',
    parse_dates=['Date'])
Secondary_market_price_USDC = peg_price_data[['Date', 'USDC_Price_USD']].copy()
Secondary_market_price_DAI = peg_price_data[['Date', 'DAI_Price_USD']].copy()
Secondary_market_price_USDT = peg_price_data[['Date', 'USDT_Price_USD']].copy()
Peg_deviation_USDC = peg_price_data[[
    'Date', 'USDC_Peg_Deviation_USD', 'USDC_Peg_Deviation_bps']].copy()
Peg_deviation_DAI = peg_price_data[[
    'Date', 'DAI_Peg_Deviation_USD', 'DAI_Peg_Deviation_bps']].copy()
Peg_deviation_USDT = peg_price_data[[
    'Date', 'USDT_Peg_Deviation_USD', 'USDT_Peg_Deviation_bps']].copy()

# Constructed redemption-pressure index
redemption_pressure_data = pd.read_csv(
    PANEL_A_DATA_FOLDER / 'Redemption_Pressure_Index_Daily.csv', parse_dates=['Date'])
Redemption_pressure_USDC = redemption_pressure_data[[
    'Date', 'USDC_Peg_Shortfall_bps', 'USDC_Net_Burn_Intensity_bps',
    'USDC_Redemption_Pressure_Index_bps']].copy()
Redemption_pressure_DAI = redemption_pressure_data[[
    'Date', 'DAI_Peg_Shortfall_bps', 'DAI_Net_Burn_Intensity_bps',
    'DAI_Redemption_Pressure_Index_bps']].copy()
Redemption_pressure_USDT = redemption_pressure_data[[
    'Date', 'USDT_Peg_Shortfall_bps', 'USDT_Net_Burn_Intensity_bps',
    'USDT_Redemption_Pressure_Index_bps']].copy()

print('Panel A variables loaded successfully')

FileNotFoundError: [Errno 2] No such file or directory: 'Panel_A_Data/USDC_USD_Daily_Price'

Panel B. Core funding and money markets (receivers)
- T-bills (Daily) (2020-latest)
    - 1 month Treasury bill yeilds 
    - 3 month Treasury bill yeilds
- Repo/rates (Daily) (2020-latest)
    - SOFR (Secured Overnight Finacing Rate)
    - 1 month OIS ( Overnight Index Swap Rate )
    - 3 month OIS
    - SFOR-OSI spread 
    - GC-Treasury (Genaral Collateral) repo spread 

In [31]:
# Panel B. Core funding and money markets (receivers)

PANEL_B_DATA_FOLDER = Path('Panel_B_Data')

# T-bills (Daily) (2020-latest)
# 1-month Treasury bill yield
t_bill_1m = pd.read_csv(PANEL_B_DATA_FOLDER / 'US_1M_Treasury_Bill_Yield.csv')

# 3-month Treasury bill yield
t_bill_3m = pd.read_csv(PANEL_B_DATA_FOLDER / 'US_3M_Treasury_Bill_Yield.csv')

# Repo/rates (Daily) (2020-latest)
# SOFR (Secured Overnight Finacing Rate) <- Error 
# 1-month SOFR OIS market rate
sofr_ois_1m = pd.read_csv(PANEL_B_DATA_FOLDER / 'USD_1M_SOFR_OIS.csv')
# 3-month SOFR OIS market rate
sofr_ois_3m = pd.read_csv(PANEL_B_DATA_FOLDER / 'USD_3M_SOFR_OIS.csv')
# SFOR-OSI spread <- Error 
# GC-Treasury (Genaral Collateral) repo spread <- Error 

print('1M T-bill:', t_bill_1m.shape)
print('3M T-bill:', t_bill_3m.shape)
print('1M SOFR OIS:', sofr_ois_1m.shape)
print('3M SOFR OIS:', sofr_ois_3m.shape)

1M T-bill: (1642, 4)
3M T-bill: (1642, 4)
1M SOFR OIS: (1711, 4)
3M SOFR OIS: (1711, 4)


In [ ]:
# DeFiLlama — requested Panel A metrics only, daily from 2020 to latest
from pathlib import Path
import requests
import pandas as pd

PANEL_A_DATA_FOLDER = Path('Panel_A_Data')
PANEL_A_DATA_FOLDER.mkdir(parents=True, exist_ok=True)
STABLECOIN_API = 'https://stablecoins.llama.fi'
TARGET_SYMBOLS = ['USDT', 'USDC', 'DAI']
ANALYSIS_START = pd.Timestamp('2020-01-01', tz='UTC')

def get_json(url, params=None):
    response = requests.get(url, params=params, timeout=90)
    response.raise_for_status()
    return response.json()

def pegged_usd(value):
    if isinstance(value, dict):
        value = value.get('peggedUSD')
    return pd.to_numeric(value, errors='coerce')

# Resolve current DeFiLlama IDs dynamically instead of hard-coding them.
catalog_response = get_json(
    f'{STABLECOIN_API}/stablecoins', params={'includePrices': 'true'})
catalog = (
    catalog_response.get('peggedAssets', [])
    if isinstance(catalog_response, dict) else catalog_response)
catalog_by_symbol = {
    str(asset.get('symbol', '')).upper(): asset
    for asset in catalog
}

defillama_price = {}
defillama_market_cap = {}
defillama_flow = {}
defillama_failures = []

for symbol in TARGET_SYMBOLS:
    try:
        asset = catalog_by_symbol.get(symbol)
        if asset is None:
            raise KeyError(f'{symbol} was not found in the DeFiLlama stablecoin catalog')
        stablecoin_id = str(asset['id'])
        print(f'Downloading {symbol} from DeFiLlama (ID {stablecoin_id})...')

        chart = get_json(
            f'{STABLECOIN_API}/stablecoincharts/all',
            params={'stablecoin': stablecoin_id},
        )
        if isinstance(chart, dict) and 'data' in chart:
            chart = chart['data']
        if not isinstance(chart, list) or not chart:
            raise ValueError('DeFiLlama returned no historical chart data')

        rows = []
        for observation in chart:
            circulating = pegged_usd(observation.get('totalCirculating'))
            market_cap = pegged_usd(observation.get('totalCirculatingUSD'))
            rows.append({
                'Date': pd.to_datetime(
                    pd.to_numeric(observation.get('date'), errors='coerce'),
                    unit='s', utc=True, errors='coerce'),
                'Circulating_Supply': circulating,
                'Market_Cap_USD': market_cap,
            })

        history = pd.DataFrame(rows).dropna(subset=['Date'])
        history['Date'] = history['Date'].dt.floor('D')
        history = (history.sort_values('Date')
                   .drop_duplicates('Date', keep='last')
                   .reset_index(drop=True))
        history['Price_USD'] = (
            history['Market_Cap_USD'] / history['Circulating_Supply'].replace(0, pd.NA))
        history['Net_Mint_Burn_Units'] = history['Circulating_Supply'].diff()
        history['Net_Flow_USD'] = history['Net_Mint_Burn_Units'] * history['Price_USD']
        history = history.loc[history['Date'] >= ANALYSIS_START].reset_index(drop=True)

        common = {
            'Symbol': symbol,
            'DeFiLlama_ID': stablecoin_id,
            'Source': 'DeFiLlama',
        }
        price = history[['Date', 'Price_USD']].copy()
        market_cap = history[['Date', 'Market_Cap_USD']].copy()
        flow = history[[
            'Date', 'Circulating_Supply', 'Net_Mint_Burn_Units', 'Net_Flow_USD']].copy()
        for frame in (price, market_cap, flow):
            frame.insert(1, 'Symbol', common['Symbol'])
            frame.insert(2, 'DeFiLlama_ID', common['DeFiLlama_ID'])
            frame.insert(3, 'Source', common['Source'])

        defillama_price[symbol] = price
        defillama_market_cap[symbol] = market_cap
        defillama_flow[symbol] = flow
        print(
            f'  saved {len(history):,} daily rows: '
            f'{history["Date"].min().date()} to {history["Date"].max().date()}')
    except Exception as error:
        defillama_failures.append({'Symbol': symbol, 'Error': str(error)})
        print(f'  FAILED: {error}')

if defillama_failures:
    print('Failed downloads:', defillama_failures)

if set(TARGET_SYMBOLS) - set(defillama_price):
    raise RuntimeError('Not all requested stablecoins downloaded; no final metric files were written.')

from functools import reduce

def merge_symbol_metric(source, value_column, output_suffix):
    frames = [
        source[symbol][['Date', value_column]].rename(
            columns={value_column: f'{symbol}_{output_suffix}'})
        for symbol in TARGET_SYMBOLS
    ]
    return reduce(lambda left, right: left.merge(right, on='Date', how='outer'), frames).sort_values('Date')

# 1) Net print/burn: positive = net minting; negative = net burning.
net_mint_burn = merge_symbol_metric(
    defillama_flow, 'Net_Mint_Burn_Units', 'Net_Mint_Burn_Units')
net_mint_burn.to_csv(
    PANEL_A_DATA_FOLDER / 'Net_Mint_Burn_USDT_USDC_DAI_Daily.csv', index=False)

# 2) Aggregate circulating supply across USDT, USDC, and DAI.
supply_by_coin = merge_symbol_metric(
    defillama_flow, 'Circulating_Supply', 'Circulating_Supply')
supply_columns = [f'{symbol}_Circulating_Supply' for symbol in TARGET_SYMBOLS]
aggregate_supply = supply_by_coin.copy()
aggregate_supply['Aggregate_Stablecoin_Supply'] = (
    aggregate_supply[supply_columns].sum(axis=1, min_count=1))
aggregate_supply.to_csv(
    PANEL_A_DATA_FOLDER / 'Aggregate_Stablecoin_Supply_Daily.csv', index=False)

# 3) Secondary-market prices and deviations from the $1 peg.
peg_deviations = merge_symbol_metric(defillama_price, 'Price_USD', 'Price_USD')
for symbol in TARGET_SYMBOLS:
    peg_deviations[f'{symbol}_Peg_Deviation_USD'] = (
        peg_deviations[f'{symbol}_Price_USD'] - 1.0)
    peg_deviations[f'{symbol}_Peg_Deviation_bps'] = (
        peg_deviations[f'{symbol}_Peg_Deviation_USD'] * 10000)
peg_deviations.to_csv(
    PANEL_A_DATA_FOLDER / 'Secondary_Prices_and_Peg_Deviations_Daily.csv', index=False)

# 4) Constructed redemption-pressure index.
# Index (bps) = downside peg deviation (bps) + net burns as bps of prior-day supply.
# It is zero when price is at/above peg and supply is not contracting; higher is more pressure.
redemption_pressure = peg_deviations[['Date']].copy()
for symbol in TARGET_SYMBOLS:
    components = peg_deviations[['Date', f'{symbol}_Price_USD']].merge(
        defillama_flow[symbol][['Date', 'Circulating_Supply', 'Net_Mint_Burn_Units']],
        on='Date', how='outer').sort_values('Date')
    peg_shortfall_bps = ((1.0 - components[f'{symbol}_Price_USD']).clip(lower=0) * 10000)
    prior_supply = components['Circulating_Supply'].shift(1)
    burn_intensity_bps = (
        (-components['Net_Mint_Burn_Units'] / prior_supply).clip(lower=0) * 10000)
    component_frame = pd.DataFrame({
        'Date': components['Date'],
        f'{symbol}_Peg_Shortfall_bps': peg_shortfall_bps,
        f'{symbol}_Net_Burn_Intensity_bps': burn_intensity_bps,
        f'{symbol}_Redemption_Pressure_Index_bps': peg_shortfall_bps + burn_intensity_bps,
    })
    redemption_pressure = redemption_pressure.merge(component_frame, on='Date', how='outer')
redemption_pressure = redemption_pressure.sort_values('Date')
redemption_pressure.to_csv(
    PANEL_A_DATA_FOLDER / 'Redemption_Pressure_Index_Daily.csv', index=False)

metric_files = [
    PANEL_A_DATA_FOLDER / 'Net_Mint_Burn_USDT_USDC_DAI_Daily.csv',
    PANEL_A_DATA_FOLDER / 'Aggregate_Stablecoin_Supply_Daily.csv',
    PANEL_A_DATA_FOLDER / 'Secondary_Prices_and_Peg_Deviations_Daily.csv',
    PANEL_A_DATA_FOLDER / 'Redemption_Pressure_Index_Daily.csv',
]
pd.DataFrame({'Created_File': [str(path) for path in metric_files]})